In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

In [ ]:
def chat(prompt, history):
    return "bananas-bro-great-for-potassium"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    return f"you said {message}, but the history is {history}"+"but i still say bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

you said 2, but the history is [{'role': 'user', 'metadata': None, 'content': '1', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'you said 1, but the history is []but i still say bananas', 'options': None}]but i still say bananas

so message carries the current input i give 
history is this part - 
[{'role': 'user', 'metadata': None, 'content': '1', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'you said 1, but the history is []but i still say bananas', 'options': None}]

its a list of dictionaries the main parts here are:

[{'role': 'user','content': '1'}, {'role': 'assistant', 'content': 'you said 1, but the history is []but i still say bananas'}]

clearly its just history of the conversation until message(2)

In [ ]:
def chat(message, history):
    history = [{"role":h['role'], "content":h['content']} for h in history]
    messages = [{"role":'system', "content":system_message}]+ history + [{"role":'user', "content":message}]

    responses = openai.chat.completions.create(model=MODEL, messages= messages)
    return responses.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch(share=True)

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content":h["content"]} for h in history]
    message = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content":message}]

    chunks = openai.chat.completions.create(model=MODEL, messages=message,stream = True)

    res =""
    for chunk in chunks:
        res+=chunk.choices[0].delta.content or ''
        yield res


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message

    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    message = [{"role":"system", "content": relevant_system_message}] + history + [{"role":"user", "content": message}]

    chunks = openai.chat.completions.create(model=MODEL, messages=message, stream=True)

    res=''
    for chunk in chunks:
        res+=chunk.choices[0].delta.content or ''
        yield res

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()